# Anchor Selector

### Procedure

1. Load SGA-2025
2. Apply size filter
3. Apply morphology selection
4. Create sampler
5. Create correct anchor save system
6. Create alert when 1,100 galaxies of each type are classified

In [1]:
# for debugging cutout_vetter
%load_ext autoreload
%autoreload 2

In [2]:
import glob
import h5py
import os

from SGA.SGA import read_sga_sample # loading SGA-2025
from astropy.table import vstack

import pandas as pd
import numpy as np

from astropy.coordinates import SkyCoord # galaxy matching
import astropy.units as u

from SGA.qa import sdss_rgb # displaying SGA-2025 images

from cutout_vetter import CutoutVetter, confusion_matrix_report # interactive image grid

In [3]:
MAX_SEP = 9.5 # arcseconds
SAMPLE_SIZE = 1000
N_COLS = 10
MORPHOLOGY_CODES = {
    "Elliptical": 20,
    "Lenticular": 0,
    "Spiral": 10,
    "Irregular": -5,
}

In [4]:
def build_cutout_index(ssl_dir, verbose=False):
    """
    Return {(region, sgaid): (hdf5_path, row_index)}
    for fast image retrieval.
    """
    files = sorted(glob.glob(os.path.join(ssl_dir, "ssl-cutouts-dr11-*.hdf5")))
    if not files:
        raise FileNotFoundError(f"No cutout files found in {ssl_dir}")

    index = {}
    for f in files:
        filename = os.path.basename(f)
        
        if "dr11-south" in filename:
            region = "dr11-south"
        elif "dr11-north" in filename:
            region = "dr11-north"
        else:
            raise ValueError(f"Could not determine region from filename: {filename}")
            
        with h5py.File(f, "r") as H:
            sgaids = H["sgaid"][:]
            if verbose:
                print(f"  {filename}: {len(sgaids):,} galaxies ({region})")
            
            for i, sgaid in enumerate(sgaids):
                key = (region, int(sgaid))
                if key in index:
                    print(f"WARNING: duplicate key found: {key}")
                index[key] = (f, i)

    print(f"Total unique (region, SGAID) pairs indexed: {len(index):,}")

    return index

def generate_SGA_2025(cutout_index):
    _, south = read_sga_sample(region="dr11-south", no_groups=True)
    _, north = read_sga_sample(region="dr11-north", no_groups=True)

    south["VI_REGION"] = "dr11-south"
    north["VI_REGION"] = "dr11-north"

    catalog = vstack([south, north]).to_pandas()
    catalog["cutout_key"] = list(zip(catalog["VI_REGION"], catalog["SGAID"].astype(int)))
    catalog = catalog[catalog["cutout_key"].isin(cutout_index)].copy()
    catalog.drop(columns="cutout_key", inplace=True)
    catalog.reset_index(drop=True, inplace=True)
    
    return catalog

def selectSizeBin(df, binNum, delim=None):
    cutout_size = 152
    pixel_value = 0.262 # arcseconds
    if not delim:
        delim = cutout_size * pixel_value / 60
    df = df[(df['D26'] > binNum*delim) & (df['D26'] <= (binNum+1)*delim)]
    print(f"Selected {len(df)} galaxies less than {delim*60} arcseconds")
    return df

# Not needed in the final version
def morphSplit(df):
    grouped = df.groupby('Galaxy_Type')

    dfs = {morph: group for morph, group in grouped}
    return dfs

def SGA_2025_Predictions(preds, SGA_2025, max_sep=MAX_SEP):

    pred_coords = SkyCoord(
        ra=preds["target_ra"].to_numpy(dtype=float),
        dec=preds["target_dec"].to_numpy(dtype=float),
        unit="deg"
    )

    sample_coords = SkyCoord(
        ra=SGA_2025["RA"].to_numpy(dtype=float),
        dec=SGA_2025["DEC"].to_numpy(dtype=float),
        unit="deg"
    )

    # For every anchor, find the nearest SGA-2025 galaxy
    idx, sep2d, _ = pred_coords.match_to_catalog_sky(sample_coords)

    max_sep_u = max_sep * u.arcsec
    good = sep2d < max_sep_u

    # SGA-2025 counterparts of successfully matched predictions
    filtered_predictions = SGA_2025.iloc[idx[good]].copy()

    # Add the prediction catalog's main_type
    filtered_predictions["Galaxy_Type"] = (preds.iloc[np.where(good)[0]]["Galaxy_Type"].to_numpy())

    print(f"Matched {good.sum()} / {len(preds)} predictions")
    print("Minimum:", sep2d.min().arcsec, "arcsec")
    print("Median :", np.median(sep2d.arcsec), "arcsec")
    print("Maximum:", sep2d.max().arcsec, "arcsec")

    return filtered_predictions, idx, sep2d

def sampleMorphologies(df, n=SAMPLE_SIZE, morph_col="Galaxy_Type"):
    sample = df.groupby(morph_col).sample(n)
    return sample
    
def lookup_by_morph(morph_label, df):
    return df[df["Galaxy_Type"] == morph_label].copy()

In [5]:
SSL_DIR = '/global/cfs/cdirs/desicollab/users/ioannis/SGA/2025/ssl'
cutout_index = build_cutout_index(SSL_DIR)
SGA_2025 = generate_SGA_2025(cutout_index)
filtered_galaxies = selectSizeBin(SGA_2025, 0)

Total unique (region, SGAID) pairs indexed: 445,693
INFO:SGA.py:363:_read_catalog: Read 395,435/395,435 GROUP_PRIMARY objects from /dvs_ro/cfs/cdirs/cosmo/work/legacysurvey/sga/2025/sample/SGA2025-beta-v1.6-dr11-south.fits
INFO:SGA.py:370:_read_catalog: Selecting 395,435/395,435 objects in region=dr11-south
INFO:SGA.py:363:_read_catalog: Read 90,504/90,504 GROUP_PRIMARY objects from /dvs_ro/cfs/cdirs/cosmo/work/legacysurvey/sga/2025/sample/SGA2025-beta-v1.6-dr11-north.fits
INFO:SGA.py:370:_read_catalog: Selecting 90,504/90,504 objects in region=dr11-north
Selected 196141 galaxies less than 39.824 arcseconds


In [6]:
predictions = pd.read_csv('/pscratch/sd/q/qshimp/Sorter/binary_classifier/sga2025/classifications/03_spiral_predictions.csv')
matched_predictions, _, _ = SGA_2025_Predictions(predictions, filtered_galaxies)
#df = morphSplit(matched_predictions)

Matched 134929 / 310903 predictions
Minimum: 0.0 arcsec
Median : 217.50685329064132 arcsec
Maximum: 9482.082311605503 arcsec


In [7]:
SAMPLE_PATH = "/pscratch/sd/q/qshimp/Sorter/anchor_selector_sample_small.csv"

matched_predictions, _, _ = SGA_2025_Predictions(predictions, filtered_galaxies)

if os.path.exists(SAMPLE_PATH):
    sample = pd.read_csv(SAMPLE_PATH)
    print(f"Loaded existing sample: {len(sample)} galaxies from {SAMPLE_PATH}")
else:
    sample = sampleMorphologies(matched_predictions)
    os.makedirs(os.path.dirname(SAMPLE_PATH), exist_ok=True)
    sample.to_csv(SAMPLE_PATH, index=False)
    print(f"Drew new sample: {len(sample)} galaxies, saved to {SAMPLE_PATH}")

Matched 134929 / 310903 predictions
Minimum: 0.0 arcsec
Median : 217.50685329064132 arcsec
Maximum: 9482.082311605503 arcsec
Loaded existing sample: 5000 galaxies from /pscratch/sd/q/qshimp/Sorter/anchor_selector_sample_small.csv


In [8]:
%matplotlib widget
vetter = CutoutVetter(
    morph_options=list(MORPHOLOGY_CODES.keys()),
    data_lookup_fn=lookup_by_morph,
    df=sample,
    cutout_index=cutout_index,
    sdss_rgb_fn=sdss_rgb,
    ncols=N_COLS,
    n_per_page=50,
    figsize_per=2,
    save_dir="/global/cfs/cdirs/desicollab/users/qshimp/anchors_small"
)

In [10]:
matrix = confusion_matrix_report(
    save_dir="/global/cfs/cdirs/desicollab/users/qshimp/anchors_small",
    morph_options=sorted(sample["Galaxy_Type"].unique()),
    username="debug"
)
matrix

['/global/cfs/cdirs/desicollab/users/qshimp/anchors_small/review_log.csv']


,Elliptical,Irregular,Lenticular,Spiral,Unknown,Bad anchor
Elliptical,227,3,84,23,0,5
Irregular,16,316,10,84,0,4
Lenticular,8,5,130,9,0,0
Spiral,0,31,17,213,0,0
Unknown,0,0,0,0,0,0


### Standings
##### Elliptical
236 Elliptical
133 Lenticular
194 Spiral
27 irregular
##### Lenticular
278 Elliptical
363 Spiral
351 Lenticular
74 irregular
##### Spiral
64 Spiral
10 Irregular
7 Lenticular
11 Elliptical
##### Irregular
229 Irregular (407 now)
468 Spiral
193 Lenticular
265 Elliptical

### Total
1,206 Elliptical
1,117 Lenticular
1,554 Spirals
1,102 Irregulars

# Next Steps
1. Combine all CSVs to master anchor catalog
2. Snip down to 4000 galaxies
3. Create public version of anchor_refinery  
    a\. Write instructions  
    b\. Create control panel  
    c\. Create user system  
    d\. Create "Validate remaining" button for each page  
    e\. Publish anchor catalog to public directory  
    f\. Create statistics for users to know how far they have come  
    g\. Fix inital load bug  
    h\. Fix too much memory use 
5. Send notebook in Slack

In [ ]:
'''
def build_master_catalog(save_dir, morph_options, output_path, selector_sample_path):
    selector_sgaids = set(pd.read_csv(selector_sample_path)["SGAID"].astype(int))

    frames = []
    for morph in morph_options:
        correct_path = os.path.join(save_dir, f"correct_{morph}.csv")
        if os.path.exists(correct_path):
            df = pd.read_csv(correct_path)
            df["Morphology"] = morph
            df["original_label"] = morph
            df["source"] = "confirmed_correct"
            df["notes"] = ""
            df["protected"] = True   # ALL correct_* rows are always protected
            frames.append(df)

        misclass_path = os.path.join(save_dir, f"misclassification_{morph}.csv")
        if os.path.exists(misclass_path):
            df = pd.read_csv(misclass_path)
            df["Morphology"] = df["alt_morphology"]
            df["original_label"] = morph
            df["source"] = "reclassified"
            df["protected"] = ~df["SGAID"].astype(int).isin(selector_sgaids)  # anchor-original only
            frames.append(df)

    master = pd.concat(frames, ignore_index=True, sort=False)
    master = master[["SGAID", "RA", "DEC", "Morphology", "original_label",
                      "notes", "timestamp", "source", "protected"]]

    dupes = master[master["SGAID"].duplicated(keep=False)]
    if len(dupes):
        print(f"WARNING: {dupes['SGAID'].nunique()} SGAID(s) appear in more than one file:")
        print(dupes.sort_values("SGAID")[["SGAID", "Morphology", "original_label", "source"]])
    master = master.drop_duplicates(subset="SGAID", keep="first")

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    master.to_csv(output_path, index=False)
    print(f"Master catalog (full): {len(master)} galaxies -> {output_path}")
    return master


def trim_master_catalog(master, target_per_type=1100, random_state=42):
    trimmed_frames = []
    for morph, group in master.groupby("Morphology"):
        protected = group[group["protected"]]
        eligible = group[~group["protected"]]

        if len(protected) >= target_per_type:
            if len(protected) > target_per_type:
                print(f"WARNING: {morph} has {len(protected)} protected galaxies, "
                      f"exceeding target of {target_per_type}. Keeping all of them; "
                      f"no eligible (selector-derived) rows included.")
            trimmed_frames.append(protected)
            continue

        remaining_slots = target_per_type - len(protected)
        if len(eligible) <= remaining_slots:
            print(f"NOTE: {morph} only has {len(eligible)} eligible rows to fill "
                  f"{remaining_slots} remaining slots "
                  f"(total {len(protected) + len(eligible)} < target {target_per_type}).")
            sampled_eligible = eligible
        else:
            sampled_eligible = eligible.sample(remaining_slots, random_state=random_state)

        trimmed_frames.append(pd.concat([protected, sampled_eligible]))

    trimmed = pd.concat(trimmed_frames, ignore_index=True).drop(columns="protected")
    return trimmed

def merge_bad_anchors(save_dir, morph_options, output_path):
    """Combine all bad_anchors_{morph}.csv into one file for easy future exclusion."""
    frames = []
    for morph in morph_options:
        path = os.path.join(save_dir, f"bad_anchors_{morph}.csv")
        if os.path.exists(path):
            df = pd.read_csv(path)
            df["original_label"] = morph
            frames.append(df)

    if not frames:
        print("No bad_anchors files found.")
        return pd.DataFrame(columns=["SGAID", "RA", "DEC", "original_label", "notes", "timestamp"])

    combined = pd.concat(frames, ignore_index=True, sort=False)
    combined = combined[["SGAID", "RA", "DEC", "original_label", "notes", "timestamp"]]

    dupes = combined[combined["SGAID"].duplicated(keep=False)]
    if len(dupes):
        print(f"WARNING: {dupes['SGAID'].nunique()} SGAID(s) marked bad under more than one original label:")
        print(dupes.sort_values("SGAID"))
    combined = combined.drop_duplicates(subset="SGAID", keep="first")

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    combined.to_csv(output_path, index=False)
    print(f"Bad anchors: {len(combined)} galaxies -> {output_path}")
    return combined
'''

In [ ]:
'''
save_dir = "/pscratch/sd/q/qshimp/Sorter"
morph_options = list(MORPHOLOGY_CODES.keys())
selector_sample_path = os.path.join(save_dir, "anchor_selector_sample.csv")

master = build_master_catalog(
    save_dir=save_dir,
    morph_options=morph_options,
    output_path=os.path.join(save_dir, "master_anchor_catalog_full.csv"),
    selector_sample_path=selector_sample_path,
)

bad_anchors = merge_bad_anchors(
    save_dir=save_dir,
    morph_options=morph_options,
    output_path=os.path.join(save_dir, "bad_anchors_combined.csv"),
)

master_trimmed = trim_master_catalog(master, target_per_type=1100)
master_trimmed.to_csv(os.path.join(save_dir, "master_anchor_catalog.csv"), index=False)

print(master_trimmed["Morphology"].value_counts())
print(master.groupby(["Morphology", "source"])["protected"].value_counts())
'''